<a href="https://colab.research.google.com/github/wjsdbfwjsdbf/chch/blob/main/tiny_chess_engine_colab_v2_clear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tiny Hybrid Chess Engine — Colab 검증용

이 노트북은 **20M 완성형 이전 단계**입니다. 목표는 강한 엔진 제작이 아니라 아래 핵심 부품이 깨지지 않는지 확인하는 것입니다.

1. `python-chess` 보드 → `8×8×22` 텐서 인코딩
2. AlphaZero식 `64×73 = 4672` 착수 인덱스
3. 합법수 마스크
4. Tiny policy/value 모델
5. toy teacher 데이터셋으로 짧은 지도학습
6. Google Drive 체크포인트 저장/재개
7. 간이 PUCT MCTS 연결

처음에는 Stockfish 없이 실행됩니다. Stockfish 증류는 이 기본 파이프라인이 안정된 뒤 붙이는 것이 안전합니다.

## RUN CELL 1 — 설치

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

In [ ]:
print('===== RUN CELL 01 START =====')
# Colab 기본 PyTorch를 사용합니다. python-chess와 tqdm만 설치합니다.
!pip -q install python-chess tqdm

===== RUN CELL 01 START =====
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 94.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## RUN CELL 2 — 기본 설정, 시드, 저장 경로

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

In [ ]:
print('===== RUN CELL 02 START =====')
import os
import math
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import chess


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_TYPE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Google Drive 저장. Colab이 아니면 로컬 ./chess_engine_tiny 로 저장합니다.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/chess_engine_tiny')
except Exception as e:
    print("Google Drive mount skipped:", repr(e))
    BASE_DIR = Path('./chess_engine_tiny')

BASE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_LATEST = BASE_DIR / 'checkpoint_latest.pt'
CKPT_BEST = BASE_DIR / 'checkpoint_best.pt'
print("BASE_DIR:", BASE_DIR.resolve())


@dataclass
class Config:
    # 입력/정책 공간
    in_channels: int = 22
    board_size: int = 8
    action_planes: int = 73
    action_size: int = 64 * 73

    # tiny 모델. Colab T4에서도 빠르게 검증되는 크기.
    channels: int = 64
    res_blocks: int = 3
    transformer_layers: int = 1
    transformer_heads: int = 4
    transformer_ff: int = 256
    dropout: float = 0.05

    # toy 학습 설정
    num_train_positions: int = 10000
    num_val_positions: int = 2000
    max_random_plies: int = 60
    batch_size: int = 128
    epochs: int = 5
    lr: float = 3e-4
    weight_decay: float = 1e-4
    value_loss_weight: float = 0.35
    num_workers: int = 0

    # 성능/안정성
    use_amp: bool = True
    use_compile: bool = False  # 처음 디버깅 때는 False 권장. 안정 후 True로 바꿔도 됩니다.

    # MCTS
    mcts_simulations: int = 96
    c_puct: float = 1.5
    dirichlet_alpha: float = 0.3
    dirichlet_weight: float = 0.25
    temperature_opening: float = 1.0
    temperature_late: float = 0.01
    opening_moves: int = 12

CFG = Config()
print(CFG)

===== RUN CELL 02 START =====
DEVICE: cuda
GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/chess_engine_tiny
Config(in_channels=22, board_size=8, action_planes=73, action_size=4672, channels=64, res_blocks=3, transformer_layers=1, transformer_heads=4, transformer_ff=256, dropout=0.05, num_train_positions=10000, num_val_positions=2000, max_random_plies=60, batch_size=128, epochs=5, lr=0.0003, weight_decay=0.0001, value_loss_weight=0.35, num_workers=0, use_amp=True, use_compile=False, mcts_simulations=96, c_puct=1.5, dirichlet_alpha=0.3, dirichlet_weight=0.25, temperature_opening=1.0, temperature_late=0.01, opening_moves=12)


## RUN CELL 3 — 보드 인코딩과 4672 착수 인코딩

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

In [ ]:
print('===== RUN CELL 03 START =====')
# 12 piece planes 순서: 백 P,N,B,R,Q,K / 흑 P,N,B,R,Q,K
PIECE_TO_PLANE: Dict[Tuple[bool, int], int] = {}
for color_offset, color in [(0, chess.WHITE), (6, chess.BLACK)]:
    for i, piece_type in enumerate([chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]):
        PIECE_TO_PLANE[(color, piece_type)] = color_offset + i

# Queen-like 방향 8개 × 거리 1~7 = 56 planes
QUEEN_DIRECTIONS: List[Tuple[int, int]] = [
    (0, 1), (1, 1), (1, 0), (1, -1),
    (0, -1), (-1, -1), (-1, 0), (-1, 1),
]
DIR_TO_INDEX = {d: i for i, d in enumerate(QUEEN_DIRECTIONS)}

# Knight 방향 8개 = planes 56~63
KNIGHT_DELTAS: List[Tuple[int, int]] = [
    (1, 2), (2, 1), (2, -1), (1, -2),
    (-1, -2), (-2, -1), (-2, 1), (-1, 2),
]
KNIGHT_TO_INDEX = {d: i for i, d in enumerate(KNIGHT_DELTAS)}

# Underpromotion 3개 기물 × 파일 변화 -1,0,1 = planes 64~72
UNDERPROMO_PIECES = [chess.KNIGHT, chess.BISHOP, chess.ROOK]
UNDERPROMO_TO_INDEX = {p: i for i, p in enumerate(UNDERPROMO_PIECES)}
FILE_DELTAS = [-1, 0, 1]
FILE_DELTA_TO_INDEX = {d: i for i, d in enumerate(FILE_DELTAS)}


def square_to_rc(square: int) -> Tuple[int, int]:
    """python-chess square -> tensor row, col. row=0 is rank 8."""
    file = chess.square_file(square)
    rank = chess.square_rank(square)
    return 7 - rank, file


def encode_board(board: chess.Board) -> torch.Tensor:
    """Board를 [22,8,8] float32 tensor로 변환합니다.

    채널 구성:
    0~11: piece planes
    12: white to move
    13~16: castling rights WK,WQ,BK,BQ
    17: en passant target square
    18: halfmove clock / 100
    19: fullmove number / 200
    20: current side is in check
    21: can claim draw by 50-move or repetition
    """
    x = torch.zeros((CFG.in_channels, 8, 8), dtype=torch.float32)

    for sq, piece in board.piece_map().items():
        plane = PIECE_TO_PLANE[(piece.color, piece.piece_type)]
        r, c = square_to_rc(sq)
        x[plane, r, c] = 1.0

    x[12].fill_(1.0 if board.turn == chess.WHITE else 0.0)
    x[13].fill_(1.0 if board.has_kingside_castling_rights(chess.WHITE) else 0.0)
    x[14].fill_(1.0 if board.has_queenside_castling_rights(chess.WHITE) else 0.0)
    x[15].fill_(1.0 if board.has_kingside_castling_rights(chess.BLACK) else 0.0)
    x[16].fill_(1.0 if board.has_queenside_castling_rights(chess.BLACK) else 0.0)

    if board.ep_square is not None:
        r, c = square_to_rc(board.ep_square)
        x[17, r, c] = 1.0

    x[18].fill_(min(board.halfmove_clock, 100) / 100.0)
    x[19].fill_(min(board.fullmove_number, 200) / 200.0)
    x[20].fill_(1.0 if board.is_check() else 0.0)
    x[21].fill_(1.0 if board.can_claim_draw() else 0.0)
    return x


def _sign(v: int) -> int:
    return 0 if v == 0 else (1 if v > 0 else -1)


def move_to_index(move: chess.Move, board: chess.Board) -> int:
    """Move -> [0,4671] index. board는 promotion 방향 판정에 필요합니다."""
    from_sq = move.from_square
    to_sq = move.to_square
    fx, fy = chess.square_file(from_sq), chess.square_rank(from_sq)
    tx, ty = chess.square_file(to_sq), chess.square_rank(to_sq)
    dx, dy = tx - fx, ty - fy

    # knight/bishop/rook underpromotion은 별도 plane
    if move.promotion in UNDERPROMO_TO_INDEX:
        forward = 1 if board.turn == chess.WHITE else -1
        if dy != forward or dx not in FILE_DELTA_TO_INDEX:
            raise ValueError(f"Invalid underpromotion geometry: {move.uci()} on {board.fen()}")
        pidx = UNDERPROMO_TO_INDEX[move.promotion]
        fidx = FILE_DELTA_TO_INDEX[dx]
        plane = 64 + pidx * 3 + fidx
        return from_sq * 73 + plane

    # knight move
    if (dx, dy) in KNIGHT_TO_INDEX:
        plane = 56 + KNIGHT_TO_INDEX[(dx, dy)]
        return from_sq * 73 + plane

    # queen-like move: rook/bishop/queen/king/pawn/queen-promotion 포함
    if dx == 0 or dy == 0 or abs(dx) == abs(dy):
        dist = max(abs(dx), abs(dy))
        if not (1 <= dist <= 7):
            raise ValueError(f"Invalid queen-like distance: {move.uci()} on {board.fen()}")
        direction = (_sign(dx), _sign(dy))
        if direction not in DIR_TO_INDEX:
            raise ValueError(f"Invalid direction: {move.uci()} on {board.fen()}")
        plane = DIR_TO_INDEX[direction] * 7 + (dist - 1)
        return from_sq * 73 + plane

    raise ValueError(f"Move is not encodable: {move.uci()} on {board.fen()}")


def index_to_move(index: int, board: chess.Board) -> chess.Move:
    """Index -> Move. promotion 복원 때문에 board가 필요합니다."""
    from_sq = index // 73
    plane = index % 73
    fx, fy = chess.square_file(from_sq), chess.square_rank(from_sq)

    promotion = None
    if plane < 56:
        dir_idx = plane // 7
        dist = (plane % 7) + 1
        dx, dy = QUEEN_DIRECTIONS[dir_idx]
        tx, ty = fx + dx * dist, fy + dy * dist
    elif plane < 64:
        dx, dy = KNIGHT_DELTAS[plane - 56]
        tx, ty = fx + dx, fy + dy
    else:
        k = plane - 64
        pidx = k // 3
        fidx = k % 3
        dx = FILE_DELTAS[fidx]
        dy = 1 if board.turn == chess.WHITE else -1
        tx, ty = fx + dx, fy + dy
        promotion = UNDERPROMO_PIECES[pidx]

    if not (0 <= tx < 8 and 0 <= ty < 8):
        return chess.Move.null()

    to_sq = chess.square(tx, ty)
    piece = board.piece_at(from_sq)
    if promotion is None and piece is not None and piece.piece_type == chess.PAWN:
        last_rank = 7 if piece.color == chess.WHITE else 0
        if ty == last_rank:
            promotion = chess.QUEEN

    return chess.Move(from_sq, to_sq, promotion=promotion)


def legal_move_mask(board: chess.Board) -> torch.Tensor:
    mask = torch.zeros(CFG.action_size, dtype=torch.bool)
    for mv in board.legal_moves:
        idx = move_to_index(mv, board)
        mask[idx] = True
    return mask


def masked_softmax(logits: torch.Tensor, mask: torch.Tensor, dim: int = -1) -> torch.Tensor:
    masked_logits = logits.masked_fill(~mask, -1e9)
    return F.softmax(masked_logits, dim=dim)

===== RUN CELL 03 START =====


## RUN CELL 4 — 인코딩 자체 검증

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

In [ ]:
print('===== RUN CELL 04 START =====')
def run_encoding_tests() -> None:
    # 시작 포지션
    b = chess.Board()
    x = encode_board(b)
    assert x.shape == (CFG.in_channels, 8, 8)
    assert int(legal_move_mask(b).sum().item()) == b.legal_moves.count() == 20

    for mv in b.legal_moves:
        idx = move_to_index(mv, b)
        mv2 = index_to_move(idx, b)
        assert mv == mv2, (mv, idx, mv2)

    # 캐슬링 포함 포지션
    b = chess.Board("r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1")
    for mv in b.legal_moves:
        idx = move_to_index(mv, b)
        mv2 = index_to_move(idx, b)
        assert mv == mv2, (mv, idx, mv2)

    # 퀸 승격 / 언더프로모션
    b = chess.Board("8/P7/8/8/8/8/7p/4K2k w - - 0 1")
    promo_moves = [m for m in b.legal_moves if m.promotion]
    assert promo_moves, "promotion moves missing"
    for mv in promo_moves:
        idx = move_to_index(mv, b)
        mv2 = index_to_move(idx, b)
        assert mv == mv2, (mv, idx, mv2)

    # 흑 언더프로모션
    b = chess.Board("4k2K/8/8/8/8/8/p7/8 b - - 0 1")
    promo_moves = [m for m in b.legal_moves if m.promotion]
    assert promo_moves, "black promotion moves missing"
    for mv in promo_moves:
        idx = move_to_index(mv, b)
        mv2 = index_to_move(idx, b)
        assert mv == mv2, (mv, idx, mv2)

    print("Encoding tests passed.")

run_encoding_tests()

===== RUN CELL 04 START =====
Encoding tests passed.


## RUN CELL 5 — Tiny Hybrid Policy/Value 모델

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

In [ ]:
print('===== RUN CELL 05 START =====')
class ResidualBlock(nn.Module):
    def __init__(self, channels: int, dropout: float = 0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = F.silu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = self.bn2(self.conv2(x))
        return F.silu(x + residual)


class TinyHybridChessNet(nn.Module):
    """Colab 검증용 소형 Hybrid ChessNet.

    Conv/ResNet으로 지역 전술 패턴을 잡고, 64칸 token Transformer로 전역 관계를 약하게 학습합니다.
    Policy는 거대한 FC가 아니라 [73,8,8] conv head로 출력합니다.
    """
    def __init__(self, cfg: Config):
        super().__init__()
        c = cfg.channels
        self.cfg = cfg

        self.stem = nn.Sequential(
            nn.Conv2d(cfg.in_channels, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
            nn.SiLU(),
        )
        self.resnet = nn.Sequential(*[ResidualBlock(c, cfg.dropout) for _ in range(cfg.res_blocks)])

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=c,
            nhead=cfg.transformer_heads,
            dim_feedforward=cfg.transformer_ff,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.transformer_layers)
        self.pos_embed = nn.Parameter(torch.zeros(1, 64, c))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.policy_head = nn.Sequential(
            nn.Conv2d(c, c, 1, bias=False),
            nn.BatchNorm2d(c),
            nn.SiLU(),
            nn.Conv2d(c, cfg.action_planes, 1),
        )
        self.value_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(c, c),
            nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(c, 1),
            nn.Tanh(),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.stem(x)
        h = self.resnet(h)

        b, c, hgt, wid = h.shape
        tokens = h.flatten(2).transpose(1, 2)  # [B,64,C]
        tokens = tokens + self.pos_embed
        tokens = self.transformer(tokens)
        h = tokens.transpose(1, 2).reshape(b, c, hgt, wid)

        # [B,73,8,8] -> flip rank axis -> [B,8,8,73] -> [B,64*73]
        # 입력 텐서는 row=rank8..rank1이지만 move index는 a1..h8 순서라서 rank axis를 뒤집습니다.
        p = self.policy_head(h)
        p = torch.flip(p, dims=[2])
        p = p.permute(0, 2, 3, 1).contiguous().view(b, -1)
        v = self.value_head(h).squeeze(-1)
        return p, v


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


model = TinyHybridChessNet(CFG).to(DEVICE)
print(f"parameters: {count_parameters(model):,}")

# torch.compile은 PyTorch 2.x + CUDA에서 안정 확인 후 켭니다.
if CFG.use_compile and hasattr(torch, "compile"):
    model = torch.compile(model, mode="reduce-overhead")
    print("torch.compile enabled")

# forward smoke test
with torch.no_grad():
    xb = encode_board(chess.Board()).unsqueeze(0).to(DEVICE)
    logits, value = model(xb)
    assert logits.shape == (1, CFG.action_size)
    assert value.shape == (1,)
print("Model smoke test passed.")

===== RUN CELL 05 START =====


/tmp/ipykernel_2106/2319933163.py:46: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.transformer_layers)


parameters: 302,026
Model smoke test passed.


## RUN CELL 6 — Toy teacher 데이터셋 생성

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

In [ ]:
print('===== RUN CELL 06 START =====')
PIECE_VALUE = {
    chess.PAWN: 1.0,
    chess.KNIGHT: 3.2,
    chess.BISHOP: 3.3,
    chess.ROOK: 5.0,
    chess.QUEEN: 9.0,
    chess.KING: 0.0,
}


def material_score_white(board: chess.Board) -> float:
    score = 0.0
    for piece in board.piece_map().values():
        v = PIECE_VALUE[piece.piece_type]
        score += v if piece.color == chess.WHITE else -v
    return score


def value_target_from_board(board: chess.Board) -> float:
    """현재 차례 입장에서의 material 기반 value. 테스트용 toy target입니다."""
    if board.is_checkmate():
        return -1.0
    if board.is_stalemate() or board.is_insufficient_material() or board.can_claim_draw():
        return 0.0
    white_score = material_score_white(board)
    stm_score = white_score if board.turn == chess.WHITE else -white_score
    return float(math.tanh(stm_score / 8.0))


def captured_piece_value(board: chess.Board, move: chess.Move) -> float:
    if board.is_en_passant(move):
        return PIECE_VALUE[chess.PAWN]
    captured = board.piece_at(move.to_square)
    return 0.0 if captured is None else PIECE_VALUE[captured.piece_type]


def center_bonus(square: int) -> float:
    f = chess.square_file(square)
    r = chess.square_rank(square)
    # 중앙 e4/d4/e5/d5 근처를 약하게 선호
    return 0.06 * (3.5 - abs(f - 3.5)) + 0.06 * (3.5 - abs(r - 3.5))


def teacher_move_score(board: chess.Board, move: chess.Move) -> float:
    """Stockfish가 아니라 파이프라인 검증용 휴리스틱 teacher."""
    score = 0.0

    if board.is_capture(move):
        attacker = board.piece_at(move.from_square)
        av = 0.1 if attacker is None else PIECE_VALUE[attacker.piece_type]
        score += 1.5 * captured_piece_value(board, move) - 0.05 * av

    if move.promotion:
        score += 0.8 * PIECE_VALUE[move.promotion]

    if board.is_castling(move):
        score += 0.35

    score += center_bonus(move.to_square)

    board.push(move)
    if board.is_checkmate():
        score += 100.0
    elif board.is_check():
        score += 0.45
    # 이동 후 자기 입장에서 material이 좋아지는지 반영
    score += 0.2 * value_target_from_board(board) * (-1.0)  # push 후 상대 차례라 부호 반전
    board.pop()

    # 결정론적 tie-breaker. 무작위 없이 항상 같은 결과가 나오게 함.
    score += (move.from_square * 0.0001 + move.to_square * 0.00001)
    return score


def teacher_best_move(board: chess.Board) -> chess.Move:
    legal = list(board.legal_moves)
    assert legal, "no legal moves"
    return max(legal, key=lambda m: teacher_move_score(board, m))


def random_board(max_plies: int, rng: random.Random) -> chess.Board:
    board = chess.Board()
    plies = rng.randint(0, max_plies)
    for _ in range(plies):
        if board.is_game_over():
            break
        legal = list(board.legal_moves)
        # 초반이 너무 말도 안 되지 않도록 teacher와 random을 섞음
        if rng.random() < 0.30:
            move = teacher_best_move(board)
        else:
            move = rng.choice(legal)
        board.push(move)
    return board


class ToyChessDataset(Dataset):
    """Colab 검증용 사전계산 데이터셋.

    __getitem__에서 매번 python-chess 연산을 반복하면 느립니다.
    따라서 FEN, 입력 텐서, legal mask, policy/value target을 init에서 미리 만듭니다.
    """
    def __init__(self, num_positions: int, max_random_plies: int, seed: int):
        self.samples = []
        rng = random.Random(seed)
        seen = set()
        pbar = tqdm(total=num_positions, desc="Generating toy positions")
        while len(self.samples) < num_positions:
            b = random_board(max_random_plies, rng)
            if b.is_game_over():
                continue
            fen = b.fen()
            if fen in seen:
                continue
            seen.add(fen)

            x = encode_board(b)
            mask = legal_move_mask(b)
            best = teacher_best_move(b)
            target_policy = move_to_index(best, b)
            target_value = value_target_from_board(b)
            # target이 반드시 legal mask 안에 있어야 합니다.
            assert bool(mask[target_policy].item()), (fen, best.uci(), target_policy)
            self.samples.append((x, mask, target_policy, target_value, fen))
            pbar.update(1)
        pbar.close()

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        x, mask, target_policy, target_value, fen = self.samples[idx]
        return (
            x,
            mask,
            torch.tensor(target_policy, dtype=torch.long),
            torch.tensor(target_value, dtype=torch.float32),
            fen,
        )


train_ds = ToyChessDataset(CFG.num_train_positions, CFG.max_random_plies, seed=100)
val_ds = ToyChessDataset(CFG.num_val_positions, CFG.max_random_plies, seed=200)

# 노트북 환경에서는 num_workers=0이 가장 예측 가능성이 높습니다.
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=(DEVICE == "cuda"))
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=(DEVICE == "cuda"))
print("train/val:", len(train_ds), len(val_ds))

===== RUN CELL 06 START =====


Generating toy positions:   0%|          | 0/10000 [00:00<?, ?it/s]

KeyboardInterrupt: 

## RUN CELL 7 — 학습, 검증, 체크포인트 저장/재개

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

In [ ]:
print('===== RUN CELL 07 START =====')
def save_checkpoint(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, best_val: float, cfg: Config) -> None:
    raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
    payload = {
        "model": raw_model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch,
        "best_val": best_val,
        "cfg": cfg.__dict__,
    }
    torch.save(payload, path)


def load_checkpoint(path: Path, model: nn.Module, optimizer: Optional[torch.optim.Optimizer] = None) -> Tuple[int, float]:
    raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
    ckpt = torch.load(path, map_location=DEVICE)
    raw_model.load_state_dict(ckpt["model"])
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    return int(ckpt.get("epoch", 0)), float(ckpt.get("best_val", float("inf")))


def batch_to_device(batch):
    x, mask, target_policy, target_value, fens = batch
    return (
        x.to(DEVICE, non_blocking=True),
        mask.to(DEVICE, non_blocking=True),
        target_policy.to(DEVICE, non_blocking=True),
        target_value.to(DEVICE, non_blocking=True),
        fens,
    )


def compute_loss(model: nn.Module, batch) -> Tuple[torch.Tensor, Dict[str, float]]:
    x, mask, target_policy, target_value, _ = batch_to_device(batch)
    logits, pred_value = model(x)

    # AMP/FP16 상태에서는 logits가 float16일 수 있다.
    # -1e9 마스크 값은 float16 범위를 넘으므로 loss 계산은 float32로 변환한다.
    logits_f = logits.float()
    pred_value_f = pred_value.float()
    target_value_f = target_value.float()

    masked_logits = logits_f.masked_fill(~mask, -1e9)
    policy_loss = F.cross_entropy(masked_logits, target_policy)
    value_loss = F.mse_loss(pred_value_f, target_value_f)
    loss = policy_loss + CFG.value_loss_weight * value_loss

    with torch.no_grad():
        pred_idx = masked_logits.argmax(dim=1)
        acc = (pred_idx == target_policy).float().mean().item()
        v_mae = (pred_value_f - target_value_f).abs().mean().item()

    return loss, {
        "policy_loss": float(policy_loss.detach().cpu()),
        "value_loss": float(value_loss.detach().cpu()),
        "acc": acc,
        "v_mae": v_mae,
    }


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    totals = {"loss": 0.0, "policy_loss": 0.0, "value_loss": 0.0, "acc": 0.0, "v_mae": 0.0}
    n = 0
    for batch in loader:
        loss, metrics = compute_loss(model, batch)
        bs = batch[0].shape[0]
        totals["loss"] += float(loss.detach().cpu()) * bs
        for k, v in metrics.items():
            totals[k] += v * bs
        n += bs
    return {k: v / max(n, 1) for k, v in totals.items()}


def train_model(resume: bool = True) -> None:
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda" and CFG.use_amp))
    except TypeError:
        scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda" and CFG.use_amp))

    start_epoch = 0
    best_val = float("inf")
    if resume and CKPT_LATEST.exists():
        start_epoch, best_val = load_checkpoint(CKPT_LATEST, model, optimizer)
        print(f"Resumed from epoch={start_epoch}, best_val={best_val:.4f}")

    for epoch in range(start_epoch + 1, CFG.epochs + 1):
        model.train()
        running = {"loss": 0.0, "policy_loss": 0.0, "value_loss": 0.0, "acc": 0.0, "v_mae": 0.0}
        n = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{CFG.epochs}")
        for batch in pbar:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE_TYPE, dtype=torch.float16, enabled=(DEVICE == "cuda" and CFG.use_amp)):
                loss, metrics = compute_loss(model, batch)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            bs = batch[0].shape[0]
            running["loss"] += float(loss.detach().cpu()) * bs
            for k, v in metrics.items():
                running[k] += v * bs
            n += bs
            pbar.set_postfix({k: f"{running[k]/n:.3f}" for k in ["loss", "acc", "v_mae"]})

        train_metrics = {k: v / max(n, 1) for k, v in running.items()}
        val_metrics = evaluate(model, val_loader)
        print(f"epoch={epoch} train={train_metrics} val={val_metrics}")

        save_checkpoint(CKPT_LATEST, model, optimizer, epoch, best_val, CFG)
        if val_metrics["loss"] < best_val:
            best_val = val_metrics["loss"]
            save_checkpoint(CKPT_BEST, model, optimizer, epoch, best_val, CFG)
            print("saved best:", CKPT_BEST)


train_model(resume=True)

===== RUN CELL 07 START =====
Resumed from epoch=3, best_val=2.8542


## RUN CELL 8 — 모델 착수 선택 함수

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

In [ ]:
print('===== RUN CELL 08 START =====')
@torch.no_grad()
def predict_policy_value(board: chess.Board, model: nn.Module) -> Tuple[Dict[chess.Move, float], float]:
    model.eval()
    x = encode_board(board).unsqueeze(0).to(DEVICE)
    mask = legal_move_mask(board).unsqueeze(0).to(DEVICE)
    with torch.autocast(device_type=DEVICE_TYPE, dtype=torch.float16, enabled=(DEVICE == "cuda" and CFG.use_amp)):
        logits, value = model(x)
    probs = masked_softmax(logits.float(), mask, dim=1).squeeze(0).cpu()
    priors = {}
    for mv in board.legal_moves:
        priors[mv] = float(probs[move_to_index(mv, board)].item())
    return priors, float(value.squeeze(0).float().cpu().item())


@torch.no_grad()
def select_model_move(board: chess.Board, model: nn.Module, temperature: float = 0.01) -> chess.Move:
    priors, value = predict_policy_value(board, model)
    moves = list(priors.keys())
    probs = np.array([priors[m] for m in moves], dtype=np.float64)

    if temperature <= 0.05:
        return moves[int(probs.argmax())]

    probs = np.power(np.maximum(probs, 1e-12), 1.0 / temperature)
    probs = probs / probs.sum()
    return moves[int(np.random.choice(len(moves), p=probs))]


# best checkpoint가 있으면 로드해서 확인
if CKPT_BEST.exists():
    _ = load_checkpoint(CKPT_BEST, model, optimizer=None)
    print("Loaded best checkpoint.")

b = chess.Board()
priors, v = predict_policy_value(b, model)
print("value(start):", v)
print("top moves:")
for mv, p in sorted(priors.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(mv.uci(), f"{p:.4f}")
print("selected:", select_model_move(b, model).uci())

===== RUN CELL 08 START =====
Loaded best checkpoint.
value(start): 0.00019562244415283203
top moves:
e2e4 0.6335
d2d4 0.0484
g1h3 0.0340
h2h4 0.0335
b1a3 0.0295
b1c3 0.0289
g1f3 0.0243
a2a4 0.0208
h2h3 0.0183
e2e3 0.0182
selected: e2e4


## RUN CELL 9 — 간이 PUCT MCTS

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

In [ ]:
print('===== RUN CELL 09 START =====')
class MCTSNode:
    def __init__(self, prior: float = 0.0):
        self.prior = float(prior)
        self.visit_count = 0
        self.value_sum = 0.0  # 이 노드의 side-to-move 관점 value 합
        self.children: Dict[chess.Move, 'MCTSNode'] = {}

    @property
    def q(self) -> float:
        return 0.0 if self.visit_count == 0 else self.value_sum / self.visit_count

    def expanded(self) -> bool:
        return len(self.children) > 0


def terminal_value_for_side_to_move(board: chess.Board) -> float:
    if board.is_checkmate():
        return -1.0  # 현재 차례가 체크메이트 당함
    if board.is_stalemate() or board.is_insufficient_material() or board.can_claim_draw():
        return 0.0
    return 0.0


class TinyPUCTMCTS:
    def __init__(self, model: nn.Module, cfg: Config):
        self.model = model
        self.cfg = cfg

    def expand(self, node: MCTSNode, board: chess.Board, add_noise: bool = False) -> float:
        if board.is_game_over(claim_draw=True):
            return terminal_value_for_side_to_move(board)

        priors, value = predict_policy_value(board, self.model)
        moves = list(priors.keys())
        probs = np.array([priors[m] for m in moves], dtype=np.float64)
        probs = probs / max(probs.sum(), 1e-12)

        if add_noise and len(moves) > 0:
            noise = np.random.dirichlet([self.cfg.dirichlet_alpha] * len(moves))
            probs = (1.0 - self.cfg.dirichlet_weight) * probs + self.cfg.dirichlet_weight * noise

        node.children = {m: MCTSNode(float(p)) for m, p in zip(moves, probs)}
        return value

    def select_child(self, node: MCTSNode) -> Tuple[chess.Move, MCTSNode]:
        assert node.children
        sqrt_n = math.sqrt(max(1, node.visit_count))
        best_score = -1e9
        best = None
        for move, child in node.children.items():
            # child.q는 child side-to-move 관점이므로 parent 입장에서는 -child.q
            q_parent = -child.q
            u = self.cfg.c_puct * child.prior * sqrt_n / (1 + child.visit_count)
            score = q_parent + u
            if score > best_score:
                best_score = score
                best = (move, child)
        assert best is not None
        return best

    def run(self, board: chess.Board, simulations: Optional[int] = None, add_noise: bool = True) -> Tuple[chess.Move, Dict[chess.Move, int]]:
        simulations = simulations or self.cfg.mcts_simulations
        root = MCTSNode(1.0)
        _ = self.expand(root, board, add_noise=add_noise)

        for _sim in tqdm(range(simulations), desc="MCTS", leave=False):
            scratch = board.copy(stack=False)
            node = root
            search_path = [node]

            while node.expanded() and node.children:
                move, node = self.select_child(node)
                scratch.push(move)
                search_path.append(node)
                if scratch.is_game_over(claim_draw=True):
                    break

            value = terminal_value_for_side_to_move(scratch) if scratch.is_game_over(claim_draw=True) else self.expand(node, scratch, add_noise=False)

            # leaf side-to-move 관점 value를 leaf부터 root까지 번갈아 부호 반전하며 백업
            for n in reversed(search_path):
                n.visit_count += 1
                n.value_sum += value
                value = -value

        visits = {m: child.visit_count for m, child in root.children.items()}
        move_number = board.fullmove_number
        temp = self.cfg.temperature_opening if move_number <= self.cfg.opening_moves else self.cfg.temperature_late
        selected = self.select_by_visits(visits, temp)
        return selected, visits

    @staticmethod
    def select_by_visits(visits: Dict[chess.Move, int], temperature: float) -> chess.Move:
        moves = list(visits.keys())
        counts = np.array([visits[m] for m in moves], dtype=np.float64)
        if temperature <= 0.05:
            return moves[int(counts.argmax())]
        probs = np.power(np.maximum(counts, 1e-12), 1.0 / temperature)
        probs = probs / probs.sum()
        return moves[int(np.random.choice(len(moves), p=probs))]


mcts = TinyPUCTMCTS(model, CFG)
b = chess.Board()
move, visits = mcts.run(b, simulations=32, add_noise=True)
print("MCTS selected:", move.uci())
print("top visits:")
for mv, n in sorted(visits.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(mv.uci(), n)

===== RUN CELL 09 START =====


MCTS:   0%|          | 0/32 [00:00<?, ?it/s]

MCTS selected: e2e4
top visits:
e2e4 21
g1f3 2
g2g4 2
d2d4 2
g1h3 1
g2g3 1
e2e3 1
c2c3 1
h2h4 1
b1c3 0


In [ ]:
board = chess.Board()
move, visits = mcts.run(board, simulations=96, add_noise=False)

print("MCTS selected:", move)
print("legal:", move in board.legal_moves)
print("total visits:", sum(visits.values()))
print("top visits:")

for mv, count in sorted(visits.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(mv, count)

MCTS:   0%|          | 0/96 [00:00<?, ?it/s]

MCTS selected: e2e4
legal: True
total visits: 96
top visits:
e2e4 69
d2d4 4
g1h3 3
b1a3 3
h2h4 3
g1f3 2
b1c3 2
a2a4 2
h2h3 1
g2g3 1


## RUN CELL 10 — 짧은 자기 대국 스모크 테스트

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

In [ ]:
print('===== RUN CELL 10 START =====')
def play_short_self_game(max_plies: int = 20, use_mcts: bool = False) -> chess.Board:
    board = chess.Board()
    for ply in range(max_plies):
        if board.is_game_over(claim_draw=True):
            break
        if use_mcts:
            mv, _ = mcts.run(board, simulations=CFG.mcts_simulations, add_noise=(ply < 8))
        else:
            temp = CFG.temperature_opening if ply < 2 * CFG.opening_moves else CFG.temperature_late
            mv = select_model_move(board, model, temperature=temp)
        board.push(mv)
        print(f"{ply+1:02d}. {mv.uci()}  fen={board.fen()}")
    print("result:", board.result(claim_draw=True), "game_over:", board.is_game_over(claim_draw=True))
    return board

_ = play_short_self_game(max_plies=12, use_mcts=False)

===== RUN CELL 10 START =====
01. e2e4  fen=rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1
02. e7e5  fen=rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2
03. f1a6  fen=rnbqkbnr/pppp1ppp/B7/4p3/4P3/8/PPPP1PPP/RNBQK1NR b KQkq - 1 2
04. d7d5  fen=rnbqkbnr/ppp2ppp/B7/3pp3/4P3/8/PPPP1PPP/RNBQK1NR w KQkq - 0 3
05. e4d5  fen=rnbqkbnr/ppp2ppp/B7/3Pp3/8/8/PPPP1PPP/RNBQK1NR b KQkq - 0 3
06. b7a6  fen=rnbqkbnr/p1p2ppp/p7/3Pp3/8/8/PPPP1PPP/RNBQK1NR w KQkq - 0 4
07. d2d4  fen=rnbqkbnr/p1p2ppp/p7/3Pp3/3P4/8/PPP2PPP/RNBQK1NR b KQkq - 0 4
08. e5d4  fen=rnbqkbnr/p1p2ppp/p7/3P4/3p4/8/PPP2PPP/RNBQK1NR w KQkq - 0 5
09. g2g3  fen=rnbqkbnr/p1p2ppp/p7/3P4/3p4/6P1/PPP2P1P/RNBQK1NR b KQkq - 0 5
10. f8c5  fen=rnbqk1nr/p1p2ppp/p7/2bP4/3p4/6P1/PPP2P1P/RNBQK1NR w KQkq - 1 6
11. d1g4  fen=rnbqk1nr/p1p2ppp/p7/2bP4/3p2Q1/6P1/PPP2P1P/RNB1K1NR b KQkq - 2 6
12. g8f6  fen=rnbqk2r/p1p2ppp/p4n2/2bP4/3p2Q1/6P1/PPP2P1P/RNB1K1NR w KQkq - 3 7
result: * game_over: False


## RUN CELL 11 — 다음 확장 포인트

> Colab에서는 이 제목 셀이 아니라, 바로 아래의 코드 셀 ▶을 실행하세요.

이 노트북이 정상 작동하면 다음 순서로만 확장하십시오.

1. `ToyChessDataset`을 실제 PGN/Stockfish MultiPV 데이터셋으로 교체
2. policy target을 단일 best move가 아니라 MultiPV soft label로 변경
3. value target을 material toy value가 아니라 Stockfish WDL 또는 `tanh(cp/scale)`로 변경
4. 모델을 `channels=96`, `res_blocks=4`, `transformer_layers=2` 정도로 증량
5. MCTS는 Python 정확성 검증 후 C++/Rust/ONNX 쪽으로 분리